In [ ]:
import os.path

import pandas as pd
from glob import glob
import re
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import json
import torch
import numpy as np
from bbox_annotation import Annotation, annotation_from_json
from sklearn.model_selection import train_test_split
import cv2

import matplotlib.pyplot as plt
from tool import crop_rgb

In [ ]:
np.random.seed(0)

In [ ]:
Image.MAX_IMAGE_PIXELS = None

In [ ]:
image_pattern = re.compile(r"(.*)\.([^\.]*)")

In [ ]:
data = []

In [ ]:
from thing import thing

for i in glob("dataset/images/*"):
    name = Path(i).stem

    annotation_file = Path("dataset/bbox_annotation").joinpath(f"{name}.json")

    try:
        with open(annotation_file, "r") as annotation_file:
            annotation: Annotation = annotation_from_json(json.load(annotation_file))
    except FileNotFoundError as e:
        print(f"{annotation_file} not found, skipping")
        continue

    image = Image.open(i)

    data.append(thing(Path(i), annotation))

data

In [ ]:
data_df = pd.DataFrame(data)
data_df

In [ ]:
import pickle
with open("dataset.pickle", "wb") as f:
    f.write(pickle.dumps(data_df))

In [ ]:
training_dataset, rest = train_test_split(data_df, test_size=0.3, random_state=0)
validation_dataset, testing_dataset = train_test_split(rest, test_size=0.5, random_state=0)

display(training_dataset)
display(validation_dataset)
display(testing_dataset)

In [ ]:
data_df.loc[0]

In [ ]:
example_row = data_df[data_df["name"]=="0000000009_d-582_0000020435-0001_log06_fe_5in_md"].iloc[0]

In [ ]:
im = Image.open(example_row["image"]).convert("RGB")
im

In [ ]:
sym = crop_rgb(im, example_row["lithology_legend"])
sym

In [ ]:
cv2_im = np.array(sym)[:, :, ::-1].copy()
gray = cv2.cvtColor(cv2_im, cv2.COLOR_RGB2GRAY)
_, thresh = cv2.threshold(gray, 254, 255, cv2.THRESH_BINARY_INV)
plt.imshow(gray)

In [ ]:
plt.imshow(thresh)

In [ ]:
contours, _ = cv2.findContours(thresh,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)
contours

In [ ]:
crop_rgb(im, example_row["interpreted_lithology_column"])

In [ ]:
example_row["interpreted_lithology_column"]